In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def envelope_plot(ax, x, data, color="#4C72B0"):
    """
    data: list[list[float]], where data[i] is the distribution of values at x[i].
    """
    q1, median, q3 = np.array([np.percentile(d, [25, 50, 75]) for d in data]).T
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    outlier_x = [x[i] for i, d in enumerate(data) for v in d if v < lower_fence[i] or v > upper_fence[i]]
    outlier_y = [v     for i, d in enumerate(data) for v in d if v < lower_fence[i] or v > upper_fence[i]]

    ax.fill_between(x, lower_fence, upper_fence, color=color, alpha=0.15)
    ax.fill_between(x, q1, q3, color=color, alpha=0.45)
    ax.plot(x, median, color=color, linewidth=1.5)
    if outlier_x:
        ax.scatter(outlier_x, outlier_y, color=color, s=4, alpha=0.8, linewidths=0)

In [ ]:
# --- Example usage ---
np.random.seed(42)
x = np.arange(10)
data_a = [np.random.normal(np.sin(xi), 0.3, 100).tolist() for xi in x]
data_b = [np.random.normal(np.cos(xi), 0.3, 100).tolist() for xi in x]

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
envelope_plot(axes[0], x, data_a)
envelope_plot(axes[1], x, data_b)

plt.tight_layout()
plt.show()

In [ ]:
def envelope_plot(ax, x, data, color, label):
    """
    Plot envelope-style visualization with quartiles and outliers.
    Args:
        ax: Matplotlib axis object
        x: X-axis positions (numeric array)
        data: list[list[float]], where data[i] is the distribution of values at x[i]
        color: Color for the plot
        label: Label for legend
    """
    q1, median, q3 = np.array([np.percentile(d, [25, 50, 75]) for d in data]).T
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    outlier_x = [x[i] for i, d in enumerate(data) for v in d if v < lower_fence[i] or v > upper_fence[i]]
    outlier_y = [v for i, d in enumerate(data) for v in d if v < lower_fence[i] or v > upper_fence[i]]

    ax.fill_between(x, lower_fence, upper_fence, color=color, alpha=0.15)
    ax.fill_between(x, q1, q3, color=color, alpha=0.45)
    ax.plot(x, median, color=color, linewidth=1.5, label=label)
    if outlier_x:
        ax.scatter(outlier_x, outlier_y, color=color, s=20, alpha=0.8, linewidths=0)


def draw_stacked_boxplots(
    data_top: list[list[list[float]]],
    data_bottom: list[list[list[float]]],
    x_labels: list[str],
    y_label_top: str,
    y_label_bottom: str,
    group_labels: list[str],
    x_label: str,
):
    """
    Draw stacked plots with boxplots on top and envelope plots on bottom.
    """
    n_groups = len(data_top)
    n_positions = len(x_labels)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    # Calculate box width and spacing
    box_width = 0.8 / n_groups
    group_offset = np.arange(n_groups) - (n_groups - 1) / 2
    group_offset = group_offset * box_width
    
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

    # --- Top plot: Boxplots ---
    for i, (group, label) in enumerate(zip(data_top, group_labels)):
        positions = np.arange(n_positions) + group_offset[i]
        bp = ax_top.boxplot(
            group,
            positions=positions,
            widths=box_width * 0.9,
            patch_artist=True,
            flierprops=dict(
                marker='.',
                markerfacecolor=colors[i],
                markeredgecolor=colors[i],
                markersize=6,
                linestyle='none'
            ),
            medianprops=dict(color=colors[i], linewidth=1.5),
            boxprops=dict(facecolor=colors[i], edgecolor=colors[i], alpha=0.7),
            whiskerprops=dict(color=colors[i]),
            capprops=dict(color=colors[i]),
        )
        # Add legend entry
        ax_top.plot([], [], color=colors[i], linewidth=8, alpha=0.7, label=label)

    ax_top.set_ylabel(y_label_top, fontsize=11)
    ax_top.tick_params(bottom=False)
    ax_top.grid(axis='both', linestyle='--', alpha=0.4)
    ax_top.legend(loc='upper left', fontsize=10, frameon=False)

    # --- Bottom plot: Envelope plots ---
    x_numeric = np.arange(n_positions)
    for i, (group, label) in enumerate(zip(data_bottom, group_labels)):
        envelope_plot(ax_bot, x_numeric, group, colors[i], label)

    ax_bot.set_ylabel(y_label_bottom, fontsize=11)
    ax_bot.set_xlabel(x_label, fontsize=11)
    ax_bot.set_xticks(x_numeric)
    ax_bot.set_xticklabels(x_labels)
    ax_bot.grid(axis='both', linestyle='--', alpha=0.4)
    ax_bot.set_ylim(0, 9)

    plt.tight_layout()
    plt.savefig('scale_trend.jpg', dpi=300, bbox_inches='tight')
    plt.show()